In [2]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

from google import genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)


model = "gemini-2.0-flash-live-001"

In [3]:
from google.genai.types import (
    LiveConnectConfig,
    SpeechConfig,
    VoiceConfig,
    PrebuiltVoiceConfig
)

voice_name = "Aoede"


audio_config = LiveConnectConfig(
    response_modalities=["AUDIO"],
    speech_config=SpeechConfig(
        voice_config=VoiceConfig(
            prebuilt_voice_config=PrebuiltVoiceConfig(voice_name=voice_name)
        )
    ),
)

In [29]:
from google.genai.types import Content, Part

text_input = """Hii?""" 
                
# text_config = LiveConnectConfig(
#     response_modalities=["TEXT"]
# )

In [30]:
import soundfile as sf
import numpy as np

In [31]:
async def main():
    async with client.aio.live.connect(model=model, config=audio_config) as session:
        # Send a text prompt to Gemini
        await session.send_client_content(
            turns=Content(role="user", parts=[Part(text=text_input)]),
            turn_complete=True
        )

        # Collect audio chunks
        audio_data = []

        async for message in session.receive():
            if message.server_content and message.server_content.model_turn:
                for part in message.server_content.model_turn.parts:
                    if part.inline_data and part.inline_data.data:
                        chunk = np.frombuffer(part.inline_data.data, dtype=np.int16)
                        audio_data.append(chunk)

        # Save the audio if received
        if audio_data:
            print("✅ Received audio data, saving as 'output.wav'")
            audio = np.concatenate(audio_data)
            sf.write("output.wav", audio, samplerate=24000)
        else:
            print("⚠️ No audio data received.")
            


In [32]:
import nest_asyncio
nest_asyncio.apply()
await main()

✅ Received audio data, saving as 'output.wav'
